# 🇻🇳 Fine-tune CNN GTSRB → biển báo Việt Nam (Colab)

**Pipeline 2-stage transfer learning** từ model `custom_cnn_v1.keras` (43 lớp GTSRB Đức)
sang dataset biển báo VN (Roboflow).

**Yêu cầu trước khi chạy:**
1. `Runtime → Change runtime type → GPU (T4)`.
2. Upload **mã nguồn** dự án (zip code, không cần data) — mục 4.
3. Upload `models/custom_cnn_v1.keras` lên Google Drive — mục 5.
4. Có **Roboflow API key** (free, lấy ở https://app.roboflow.com/settings/api) — mục 7.

**Output cuối cùng** (lưu vào Drive, tải về local sau):
- `custom_cnn_vn_v1.keras` — model đã fine-tune
- `labels_vn.json` — nhãn N lớp VN (theo dataset Roboflow)
- `metrics_vn.json`, `classification_report_vn.txt`, `training_curves_vn.png`,
  `confusion_matrix_vn.png`, `comparison_gtsrb_vs_vn.md`


## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

Cần Drive để: (a) đọc pretrained `custom_cnn_v1.keras`,
(b) lưu artifacts cuối cùng (model VN + reports).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/

## 3. Cấu hình (BẠN CẦN ĐIỀN ROBOFLOW INFO)

### Cách lấy 4 thông số Roboflow:
1. Vào https://universe.roboflow.com/ → search "Vietnam Traffic Sign"
2. Chọn 1 dataset (vd "Vietnam Traffic Sign Detection" của workspace `vietnam-traffic-sign-detection`)
3. Click **Download Dataset** → format **YOLOv8** → tab **Show Download Code**
4. Copy 4 giá trị: `api_key`, `workspace`, `project`, `version` từ snippet sinh ra.

In [ ]:
import os, json, shutil, random, sys
from pathlib import Path

# ============== ROBOFLOW (BẮT BUỘC ĐIỀN) ==============
ROBOFLOW_API_KEY = "PASTE_YOUR_API_KEY_HERE"
ROBOFLOW_WORKSPACE = "vietnam-traffic-sign-detection"
ROBOFLOW_PROJECT = "vietnam-traffic-sign-detection-2i2j8"
ROBOFLOW_VERSION = 6
# =======================================================

# Pretrained GTSRB (upload trước lên Drive)
PRETRAINED_DRIVE_PATH = "/content/drive/MyDrive/custom_cnn_v1.keras"

# Drive output (artifacts cuối cùng)
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/finetune_vn")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters (khớp với src/config.py + best practice transfer learning)
IMG_SIZE = 48
BATCH_SIZE = 64
SEED = 42
PADDING_RATIO = 0.05  # giữ giống GTSRB prep
STAGE1_EPOCHS = 12
STAGE1_LR = 1e-3
STAGE2_EPOCHS = 25
STAGE2_LR = 1e-4
EARLY_STOP_PATIENCE = 8
LR_REDUCE_PATIENCE = 3
CLASS_WEIGHT_CAP = 3.0  # tăng từ 2.0 (GTSRB) vì VN imbalance nặng hơn

# Paths trong Colab
PROJECT_DIR = Path("/content/project")
RAW_DIR = Path("/content/vn_raw")
CROPPED_DIR = Path("/content/processed_vn")
MODELS_DIR = Path("/content/models")
REPORTS_DIR = Path("/content/reports")
FIGURES_DIR = REPORTS_DIR / "figures"
for d in (MODELS_DIR, REPORTS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("✅ Config loaded")
print(f"   Stage1: {STAGE1_EPOCHS} epochs @ LR={STAGE1_LR}")
print(f"   Stage2: {STAGE2_EPOCHS} epochs @ LR={STAGE2_LR}")

## 4. Upload mã nguồn dự án (zip không kèm data)

Để tái dùng `src.preprocessing.augment`, `src.data_loader.make_tf_dataset`,
`src.train.compute_class_weights` mà KHÔNG sửa code local.

### Cách đóng gói ở local (PowerShell Windows):
```powershell
Compress-Archive -Path src, requirements.txt, data\classes.txt `
                 -DestinationPath final_project_code.zip -Force
```

### Cell dưới: upload zip lên Colab và giải nén

In [ ]:
from google.colab import files
uploaded = files.upload()  # chọn final_project_code.zip
zip_name = list(uploaded.keys())[0]
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
!unzip -q -o {zip_name} -d {PROJECT_DIR}
sys.path.insert(0, str(PROJECT_DIR))
print("✅ Project unzipped:")
!ls {PROJECT_DIR}

## 5. Copy pretrained `custom_cnn_v1.keras` từ Drive

**Lưu ý:** Trước khi chạy notebook, bạn upload file `models/custom_cnn_v1.keras`
(local) lên Drive tại path `/content/drive/MyDrive/custom_cnn_v1.keras`.

In [ ]:
assert Path(PRETRAINED_DRIVE_PATH).exists(), \
    f"Không tìm thấy {PRETRAINED_DRIVE_PATH}. Hãy upload custom_cnn_v1.keras lên Drive trước."

PRETRAINED_LOCAL = MODELS_DIR / "custom_cnn_v1.keras"
shutil.copy(PRETRAINED_DRIVE_PATH, PRETRAINED_LOCAL)
print(f"✅ Copied pretrained → {PRETRAINED_LOCAL}")
print(f"   Size: {PRETRAINED_LOCAL.stat().st_size / 1e6:.2f} MB")

## 6. Cài đặt dependencies

In [ ]:
!pip install -q roboflow
!pip install -q -r {PROJECT_DIR}/requirements.txt
print("✅ Deps installed")

## 7. Tải dataset VN từ Roboflow

Format YOLOv8 (image + .txt bbox normalized).

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download("yolov8", location=str(RAW_DIR))

print(f"\n✅ Dataset tải về: {RAW_DIR}")
!ls {RAW_DIR}

## 8. EDA nhanh + đọc class names

Roboflow YOLOv8 export có cấu trúc:
```
RAW_DIR/
  train/{images,labels}/
  valid/{images,labels}/
  test/{images,labels}/
  data.yaml   # chứa list class names
```

In [ ]:
import yaml
from collections import Counter, defaultdict

# Tìm data.yaml (Roboflow lưu trong RAW_DIR hoặc subdir)
yaml_candidates = list(RAW_DIR.rglob("data.yaml"))
assert yaml_candidates, f"Không tìm thấy data.yaml trong {RAW_DIR}"
DATA_YAML = yaml_candidates[0]
DATASET_ROOT = DATA_YAML.parent
print(f"Dataset root: {DATASET_ROOT}")

with open(DATA_YAML) as f:
    yml = yaml.safe_load(f)

CLASS_NAMES = yml["names"]
NUM_CLASSES_VN = len(CLASS_NAMES)
print(f"✅ Số lớp VN: {NUM_CLASSES_VN}")
print(f"   5 lớp đầu: {CLASS_NAMES[:5]}")
print(f"   5 lớp cuối: {CLASS_NAMES[-5:]}")

# Đếm bbox/lớp ở từng split
def count_split(split_name: str):
    lbl_dir = DATASET_ROOT / split_name / "labels"
    if not lbl_dir.exists():
        return Counter()
    cnt = Counter()
    for txt in lbl_dir.glob("*.txt"):
        for line in txt.read_text().splitlines():
            parts = line.split()
            if parts:
                cnt[int(parts[0])] += 1
    return cnt

for sp in ["train", "valid", "test"]:
    c = count_split(sp)
    if c:
        print(f"\n{sp}: {sum(c.values())} bbox / {len(c)} lớp")
        rare = sorted(c.items(), key=lambda x: x[1])[:3]
        print(f"   3 lớp ít nhất: {[(CLASS_NAMES[i], n) for i, n in rare]}")

## 9. Convert YOLO bbox → classification crops (padding 5%)

Logic crop **giống hệt** GTSRB pipeline (`src.prepare_data.crop_with_roi`):
1. Đọc image + .txt
2. Mỗi bbox: convert (cx, cy, w, h) normalized → (x1, y1, x2, y2) absolute pixel
3. Pad 5% mỗi cạnh, clamp vào ảnh
4. Lưu crop vào `processed_vn/<split>/<class_idx>_<class_name>/<basename>_<bbox_idx>.png`

In [ ]:
from PIL import Image
from tqdm import tqdm

def yolo_to_xyxy(cx, cy, w, h, W, H):
    x1 = (cx - w / 2) * W
    y1 = (cy - h / 2) * H
    x2 = (cx + w / 2) * W
    y2 = (cy + h / 2) * H
    return x1, y1, x2, y2

def safe_folder_name(idx: int, name: str) -> str:
    safe = "".join(c if c.isalnum() else "_" for c in name)
    return f"{idx:02d}_{safe}"

def process_split_yolo(src_split: str, dst_split: str):
    img_dir = DATASET_ROOT / src_split / "images"
    lbl_dir = DATASET_ROOT / src_split / "labels"
    if not img_dir.exists():
        print(f"⚠️ Bỏ qua {src_split} (không tồn tại)")
        return 0, 0
    out_root = CROPPED_DIR / dst_split
    n_crops, n_skip = 0, 0
    for img_path in tqdm(sorted(img_dir.glob("*")), desc=f"[{src_split}->{dst_split}]"):
        if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp"}:
            continue
        txt_path = lbl_dir / f"{img_path.stem}.txt"
        if not txt_path.exists() or txt_path.stat().st_size == 0:
            n_skip += 1
            continue
        try:
            img = Image.open(img_path).convert("RGB")
        except Exception:
            n_skip += 1
            continue
        W, H = img.size
        for i, line in enumerate(txt_path.read_text().splitlines()):
            parts = line.split()
            if len(parts) < 5:
                continue
            cls_idx = int(parts[0])
            cx, cy, w, h = map(float, parts[1:5])
            x1, y1, x2, y2 = yolo_to_xyxy(cx, cy, w, h, W, H)
            pw, ph = (x2 - x1) * PADDING_RATIO, (y2 - y1) * PADDING_RATIO
            x1, y1 = max(0, x1 - pw), max(0, y1 - ph)
            x2, y2 = min(W, x2 + pw), min(H, y2 + ph)
            if x2 - x1 < 8 or y2 - y1 < 8:
                continue
            crop = img.crop((x1, y1, x2, y2))
            cls_dir = out_root / safe_folder_name(cls_idx, CLASS_NAMES[cls_idx])
            cls_dir.mkdir(parents=True, exist_ok=True)
            crop.save(cls_dir / f"{img_path.stem}_{i}.png", format="PNG")
            n_crops += 1
    return n_crops, n_skip

if CROPPED_DIR.exists():
    shutil.rmtree(CROPPED_DIR)

# Roboflow split: train/valid/test → processed_vn/train/val/test
for src, dst in [("train", "train"), ("valid", "val"), ("test", "test")]:
    n, s = process_split_yolo(src, dst)
    print(f"  {src}->{dst}: {n} crops, {s} skipped")

print("\n✅ Cropping done. Folders:")
for sp in ["train", "val", "test"]:
    sp_dir = CROPPED_DIR / sp
    if sp_dir.exists():
        n_files = sum(1 for _ in sp_dir.rglob("*.png"))
        n_classes = len(list(sp_dir.iterdir()))
        print(f"   {sp}: {n_files} ảnh, {n_classes} lớp")

## 10. Sinh `labels_vn.json` + xử lý split rỗng

Một số dataset Roboflow không có split `valid` riêng → nếu rỗng,
tự tách 10% từ train (stratified, seed cố định).

In [ ]:
# Nếu val rỗng, split lại từ train (stratified)
val_dir = CROPPED_DIR / "val"
need_split_val = (not val_dir.exists()) or sum(1 for _ in val_dir.rglob("*.png")) < NUM_CLASSES_VN

if need_split_val:
    print("⚠️ Val rỗng/quá ít → tách 10% từ train (stratified)")
    train_dir = CROPPED_DIR / "train"
    val_dir.mkdir(parents=True, exist_ok=True)
    rng = random.Random(SEED)
    for cls_dir in sorted(train_dir.iterdir()):
        files = sorted(cls_dir.glob("*.png"))
        rng.shuffle(files)
        n_val = max(1, int(len(files) * 0.10))
        val_cls_dir = val_dir / cls_dir.name
        val_cls_dir.mkdir(parents=True, exist_ok=True)
        for f in files[:n_val]:
            shutil.move(str(f), str(val_cls_dir / f.name))
    print("✅ Split val xong")

# Sinh labels_vn.json (sort theo folder để cố định thứ tự lớp)
class_folders = sorted([p.name for p in (CROPPED_DIR / "train").iterdir() if p.is_dir()])
labels_meta = []
for folder in class_folders:
    idx = int(folder.split("_")[0])
    qcvn_code = CLASS_NAMES[idx]
    labels_meta.append({
        "folder": folder,
        "qcvn_code": qcvn_code,
        "name_en": qcvn_code,
        "name_vi": qcvn_code,  # có thể map chi tiết sau
    })

LABELS_VN_PATH = MODELS_DIR / "labels_vn.json"
with open(LABELS_VN_PATH, "w", encoding="utf-8") as f:
    json.dump(labels_meta, f, ensure_ascii=False, indent=2)

print(f"✅ Saved {LABELS_VN_PATH}")
print(f"   {len(labels_meta)} lớp, vd: {labels_meta[0]}")

## 11. Build tf.data pipelines

Tái dùng `src.preprocessing.augment` (tương đương training GTSRB:
xoay 12°, brightness/contrast/saturation 0.85–1.15).

In [ ]:
import tensorflow as tf
import numpy as np
from src.preprocessing import augment as augment_fn  # tái dùng từ codebase

def gather(split: str):
    sp_dir = CROPPED_DIR / split
    paths, labels = [], []
    for i, folder in enumerate(class_folders):
        cdir = sp_dir / folder
        if not cdir.exists():
            continue
        for p in cdir.glob("*.png"):
            paths.append(str(p))
            labels.append(i)
    return np.array(paths), np.array(labels)

def decode(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

def make_ds(paths, labels, shuffle=False, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(min(len(paths), 2048), seed=SEED)
    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(lambda x, y: (augment_fn(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

x_tr, y_tr = gather("train")
x_val, y_val = gather("val")
x_te, y_te = gather("test")
print(f"Train: {len(x_tr)} | Val: {len(x_val)} | Test: {len(x_te)}")

train_ds = make_ds(x_tr, y_tr, shuffle=True, augment=True)
val_ds = make_ds(x_val, y_val)
test_ds = make_ds(x_te, y_te)
print("✅ tf.data pipelines ready")

## 12. Tính class_weight (cap tại 3.0 — VN imbalance nặng hơn GTSRB)

In [ ]:
from collections import Counter

def compute_class_weights(y, n_classes, cap=CLASS_WEIGHT_CAP):
    counts = Counter(int(v) for v in y)
    total = sum(counts.values())
    raw = {i: total / (n_classes * counts.get(i, 1)) for i in range(n_classes)}
    return {i: min(w, cap) for i, w in raw.items()}

class_weight = compute_class_weights(y_tr, NUM_CLASSES_VN)
print(f"Class weights: min={min(class_weight.values()):.2f}, "
      f"max={max(class_weight.values()):.2f}")

## 13. Build finetune model (transfer learning)

**Chiến lược:**
- Load pretrained `custom_cnn_v1.keras` (43 lớp GTSRB)
- Cắt từ layer `gap` (giữ toàn bộ feature extractor)
- Thay Dense head mới với N lớp VN
- Stage 1: **Freeze** block1+2+3 → chỉ train head mới
- Stage 2: Unfreeze block3 → train cả block3 + head

In [ ]:
from tensorflow.keras import layers, Model

base = tf.keras.models.load_model(PRETRAINED_LOCAL)
print("Base model layers:", [l.name for l in base.layers])

# Cắt từ "gap"
gap_out = base.get_layer("gap").output

# Head mới — kiến trúc giống src.model._conv_block + Dense
x = layers.Dense(256, activation="relu", name="fc1_vn")(gap_out)
x = layers.BatchNormalization(name="fc1_vn_bn")(x)
x = layers.Dropout(0.5, name="fc1_vn_drop")(x)
out = layers.Dense(NUM_CLASSES_VN, activation="softmax", name="predictions_vn")(x)

finetune = Model(inputs=base.input, outputs=out, name="custom_cnn_vn_v1")

# Stage 1: freeze TOÀN BỘ backbone (block1+2+3)
for layer in finetune.layers:
    if any(layer.name.startswith(b) for b in ("block1", "block2", "block3", "image_input")):
        layer.trainable = False

trainable_params = sum(tf.size(v).numpy() for v in finetune.trainable_variables)
total_params = finetune.count_params()
print(f"\n✅ Stage 1 setup:")
print(f"   Total params: {total_params:,}")
print(f"   Trainable: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")
finetune.summary()

## 14. Stage 1 — Warm-up Dense head (~12 epochs)

LR cao (1e-3) để head mới hội tụ nhanh; backbone đứng yên không bị phá feature.

In [ ]:
import matplotlib.pyplot as plt

finetune.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=STAGE1_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy",
             tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_acc")],
)

CKPT_PATH = MODELS_DIR / "custom_cnn_vn_v1.keras"
callbacks_s1 = [
    tf.keras.callbacks.ModelCheckpoint(filepath=str(CKPT_PATH), save_best_only=True,
                                       monitor="val_loss", mode="min", verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=EARLY_STOP_PATIENCE,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                         patience=LR_REDUCE_PATIENCE, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.CSVLogger(str(REPORTS_DIR / "training_log_vn_stage1.csv")),
]

history_s1 = finetune.fit(
    train_ds, validation_data=val_ds, epochs=STAGE1_EPOCHS,
    callbacks=callbacks_s1, class_weight=class_weight, verbose=1,
)
print("\n✅ Stage 1 done")

## 15. Stage 2 — Fine-tune block3 + head (~25 epochs)

LR thấp hơn (1e-4) để không phá feature đã học. block1+2 vẫn freeze
(giữ low-level feature chung của biển báo).

In [ ]:
# Unfreeze block3
for layer in finetune.layers:
    if layer.name.startswith("block3"):
        layer.trainable = True

# Recompile (Keras yêu cầu sau khi đổi trainable)
finetune.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=STAGE2_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy",
             tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_acc")],
)

trainable_params = sum(tf.size(v).numpy() for v in finetune.trainable_variables)
print(f"Stage 2: trainable params = {trainable_params:,}")

callbacks_s2 = [
    tf.keras.callbacks.ModelCheckpoint(filepath=str(CKPT_PATH), save_best_only=True,
                                       monitor="val_loss", mode="min", verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=EARLY_STOP_PATIENCE,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                         patience=LR_REDUCE_PATIENCE, min_lr=1e-7, verbose=1),
    tf.keras.callbacks.CSVLogger(str(REPORTS_DIR / "training_log_vn_stage2.csv")),
]

history_s2 = finetune.fit(
    train_ds, validation_data=val_ds, epochs=STAGE2_EPOCHS,
    callbacks=callbacks_s2, class_weight=class_weight, verbose=1,
)
print("\n✅ Stage 2 done")

# Vẽ training curves (cả 2 stage nối lại)
def get(h, key): return h.history.get(key, [])
loss_all = get(history_s1, "loss") + get(history_s2, "loss")
val_loss_all = get(history_s1, "val_loss") + get(history_s2, "val_loss")
acc_all = get(history_s1, "accuracy") + get(history_s2, "accuracy")
val_acc_all = get(history_s1, "val_accuracy") + get(history_s2, "val_accuracy")
boundary = len(get(history_s1, "loss"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, tr, val, title in [(axes[0], loss_all, val_loss_all, "Loss"),
                            (axes[1], acc_all, val_acc_all, "Accuracy")]:
    ax.plot(tr, label="train"); ax.plot(val, label="val")
    ax.axvline(boundary - 0.5, color="red", linestyle="--", label="Stage 1→2")
    ax.set_title(title); ax.legend(); ax.grid(True)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "training_curves_vn.png", dpi=150)
plt.show()

## 16. Đánh giá trên VN test set (classification report + confusion matrix)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load best checkpoint
finetune_best = tf.keras.models.load_model(CKPT_PATH)

vn_test_metrics = finetune_best.evaluate(test_ds, return_dict=True, verbose=1)
print(f"\nVN test: {vn_test_metrics}")

with open(REPORTS_DIR / "metrics_vn.json", "w", encoding="utf-8") as f:
    json.dump(vn_test_metrics, f, ensure_ascii=False, indent=2)

y_prob = finetune_best.predict(test_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

display_names = [m["qcvn_code"] for m in labels_meta]
text_report = classification_report(y_te, y_pred, target_names=display_names,
                                     digits=4, zero_division=0)
print(text_report)
with open(REPORTS_DIR / "classification_report_vn.txt", "w", encoding="utf-8") as f:
    f.write(text_report)

# Confusion matrix
cm = confusion_matrix(y_te, y_pred)
fig, ax = plt.subplots(figsize=(max(8, NUM_CLASSES_VN * 0.35),
                                 max(6, NUM_CLASSES_VN * 0.30)))
sns.heatmap(cm, annot=False, cmap="Blues",
            xticklabels=display_names, yticklabels=display_names, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.xticks(rotation=90); plt.yticks(rotation=0)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "confusion_matrix_vn.png", dpi=150)
plt.show()

## 17. So sánh: GTSRB-only vs Finetuned (ablation)

3 thí nghiệm:
1. **Baseline GTSRB**: model gốc (43 lớp Đức) đánh giá trên VN test → cho thấy
   visual/domain shift nghiêm trọng đến đâu
2. **Finetuned trên VN test**: kết quả chính
3. (Tùy chọn) **Finetuned trên GTSRB test**: kiểm tra catastrophic forgetting
   — yêu cầu test_GTSRB sẵn trên Colab; ở đây bỏ qua nếu chưa có

In [ ]:
from collections import defaultdict

# (1) Baseline GTSRB on VN test — top-1 vô nghĩa (43 vs N lớp khác nhau)
#     → đo "best-effort": dự đoán argmax 43 lớp + confidence trung bình
baseline = tf.keras.models.load_model(PRETRAINED_LOCAL)
baseline_probs = baseline.predict(test_ds, verbose=1)
baseline_conf = float(np.mean(np.max(baseline_probs, axis=1)))

# (2) Finetuned trên VN test — đã có vn_test_metrics ở mục 16

comparison = {
    "baseline_gtsrb_on_vn_test": {
        "note": "Model gốc 43 lớp, không thể tính accuracy trực tiếp do label space khác. "
                "Đo confidence trung bình của argmax → thấp = model 'không chắc chắn' khi gặp biển VN.",
        "mean_max_confidence": round(baseline_conf, 4),
        "num_test_images": int(len(y_te)),
    },
    "finetuned_on_vn_test": {
        "accuracy": round(vn_test_metrics.get("accuracy", 0.0), 4),
        "top3_accuracy": round(vn_test_metrics.get("top3_acc", 0.0), 4),
        "loss": round(vn_test_metrics.get("loss", 0.0), 4),
        "num_classes": NUM_CLASSES_VN,
        "num_test_images": int(len(y_te)),
    },
}

md_lines = [
    "# So sánh: Baseline GTSRB vs Finetuned VN", "",
    "## 1. Baseline GTSRB (chưa finetune) trên VN test",
    f"- Mean max confidence: **{baseline_conf:.4f}**",
    f"- Số ảnh test VN: {len(y_te)}",
    "- *Lưu ý*: Không tính được accuracy trực tiếp do label space khác (43 vs "
    f"{NUM_CLASSES_VN}). Confidence thấp = model 'lúng túng' khi gặp biển VN.", "",
    "## 2. Finetuned trên VN test",
    f"- Accuracy: **{vn_test_metrics.get('accuracy', 0):.4f}**",
    f"- Top-3 accuracy: **{vn_test_metrics.get('top3_acc', 0):.4f}**",
    f"- Loss: {vn_test_metrics.get('loss', 0):.4f}",
    f"- Số lớp VN: {NUM_CLASSES_VN}", "",
    "## 3. Kết luận",
    "Fine-tune 2-stage cải thiện rõ rệt khả năng nhận diện biển VN so với "
    "model GTSRB gốc, nhờ tận dụng feature extractor block1+2 (kiến thức chung "
    "về biển báo theo Vienna 1968) và học lại block3 + head cho QCVN 41:2019.",
]
(REPORTS_DIR / "comparison_gtsrb_vs_vn.md").write_text("\n".join(md_lines), encoding="utf-8")
with open(REPORTS_DIR / "comparison_gtsrb_vs_vn.json", "w", encoding="utf-8") as f:
    json.dump(comparison, f, ensure_ascii=False, indent=2)
print("✅ Saved comparison_gtsrb_vs_vn.{md,json}")
print(json.dumps(comparison, indent=2, ensure_ascii=False))

## 18. Export artifacts ra Drive

Lưu tất cả vào `/content/drive/MyDrive/finetune_vn/` để tải về local.

In [ ]:
ARTIFACTS = [
    (CKPT_PATH, "custom_cnn_vn_v1.keras"),
    (LABELS_VN_PATH, "labels_vn.json"),
    (REPORTS_DIR / "metrics_vn.json", "metrics_vn.json"),
    (REPORTS_DIR / "classification_report_vn.txt", "classification_report_vn.txt"),
    (REPORTS_DIR / "comparison_gtsrb_vs_vn.md", "comparison_gtsrb_vs_vn.md"),
    (REPORTS_DIR / "comparison_gtsrb_vs_vn.json", "comparison_gtsrb_vs_vn.json"),
    (REPORTS_DIR / "training_log_vn_stage1.csv", "training_log_vn_stage1.csv"),
    (REPORTS_DIR / "training_log_vn_stage2.csv", "training_log_vn_stage2.csv"),
    (FIGURES_DIR / "training_curves_vn.png", "training_curves_vn.png"),
    (FIGURES_DIR / "confusion_matrix_vn.png", "confusion_matrix_vn.png"),
]
for src, name in ARTIFACTS:
    if src.exists():
        shutil.copy(src, DRIVE_OUTPUT_DIR / name)
        print(f"  ✅ {name} ({src.stat().st_size / 1e6:.2f} MB)")
    else:
        print(f"  ⚠️ Skip (missing): {src}")

print(f"\n📁 Tất cả ở: {DRIVE_OUTPUT_DIR}")
!ls -lh {DRIVE_OUTPUT_DIR}

## 19. Hướng dẫn deploy về local (zero code change)

### Bước 1: Tải các file sau từ Drive về local
| Drive | Local |
|---|---|
| `finetune_vn/custom_cnn_vn_v1.keras` | `models/custom_cnn_vn_v1.keras` |
| `finetune_vn/labels_vn.json` | `models/labels_vn.json` |
| `finetune_vn/metrics_vn.json` + reports | `reports/...` |

### Bước 2 (CHỌN 1 trong 2):

**Option A — Zero code change (transparent VN model):**
```powershell
# Backup file GTSRB cũ trước
Copy-Item models/custom_cnn_v1.keras models/custom_cnn_v1.gtsrb.keras
Copy-Item models/labels.json         models/labels.gtsrb.json

# Thay bằng file VN (đổi tên)
Move-Item models/custom_cnn_vn_v1.keras models/custom_cnn_v1.keras -Force
Move-Item models/labels_vn.json         models/labels.json         -Force
```
→ App Streamlit + realtime camera tự load VN model, **0 dòng code thay đổi**.

**Option B — Giữ song song (cần sửa 2 dòng):**
- Edit `src/config.py`: `MODEL_NAME = "custom_cnn_vn_v1"`
- Edit `src/data_loader.load_labels()` đọc `labels_vn.json` thay vì `labels.json`

→ Có thể switch giữa 2 model bằng cách đổi config.

### Bước 3: Chạy app
```powershell
streamlit run app/streamlit_app.py
```